# 构建定时仓库审阅器

## 引言

把反复出现的审阅苦活变成由你掌控的无人值守任务。即使是审阅充分的仓库，在两次审阅之间也会发生变化，而这些变化可能带来以往审阅未曾标识的问题。本示例把这项工作交给智能体。

在你选定的时间计划上，智能体扫描你的仓库，返回程序可读的结论与发现项。跨次运行的连续性来自恢复会话（resumed sessions），这是本示例所围绕的 Agent SDK 特性。恢复只是定时审阅器的一种设计选择。被恢复的会话会把上一轮的全部工作带到下一轮，无需另建存储。每次运行时，此前的发现项和审阅者读过的内容都已加载，而不是从零开始。*何时恢复、何时重新开始*一节说明各选择分别适合哪些工作。

首次运行时对仓库做一次完整的基线审阅。之后每次运行都恢复上一次的会话，并从智能体已报告过的发现项出发。每个周期你只需阅读一段简短的后续报告，而每份后续报告都会复述上一次审阅的发现项，以此证明连续性。本笔记本手动执行前两个周期，最后把同一个审阅器排上时间计划。

## 你将学到什么

完成本示例后，你将能够：

- 在 `permission_mode="dontAsk"` 下无人值守地运行一个有边界、只读的审阅智能体，并为调度器设置 `max_turns` 与 `max_budget_usd`
- 通过 `ClaudeAgentOptions(resume=...)` 恢复会话，并断言 `output_format` 模式响应中的 `RESUME-LINK` 字段，以此证明跨次运行的连续性；其中已修复的发现项会移入 `resolved`，新植入的缺陷会被捕获
- 用 `scheduled_review.py` 把审阅器交给 cron，输出可用 grep 检索的 `VERDICT` 行与完成行，并设置一条范围收窄的 `except ResultError` 路径，以非零状态退出

## 何时使用本示例

如果要做的是能记住上一轮的周期性审阅，例如依赖审计或文档新鲜度巡检，就使用本示例。调度器由你自己运行，Claude Code 提供只读工具、权限规则和可恢复的会话。

如果你不想自己运行调度器或基础设施，下面某个托管方案可能更合适：

- [Claude Code routines](https://code.claude.com/docs/en/routines) 适合希望按计划审阅自己 GitHub 仓库的独立开发者。你只需配置一次提示词、仓库和连接器，Claude 就在托管基础设施上运行该例行任务。Routines 处于研究预览阶段，并在 Claude 订阅方案下运行。
- [Managed Agents](https://platform.claude.com/docs/en/managed-agents/overview) 适合希望通过 Claude API 配置托管服务的情况。它托管智能体循环及其工作区，而[定时部署](https://platform.claude.com/docs/en/managed-agents/scheduled-deployments)会按你设定的 cron 节奏启动会话。Managed Agents 处于 beta 阶段，被审阅的仓库会进入托管的会话工作区。

如果你想自己运行调度器和审阅环境，就使用本示例。

## 前置条件

在跟随本指南之前，请确认你具备：

**必备知识：**

* Python 基础：熟悉 async/await、函数和基本数据结构
* 对智能体模式有基本理解：建议阅读[构建高效智能体](https://www.anthropic.com/engineering/building-effective-agents)
* 对 Agent SDK 的 `query()` 函数有基本了解：如果你刚接触该 SDK，可参见[一行代码的研究智能体](../00_The_one_liner_research_agent.ipynb)

**必备工具：**

* Python 3.11 或更高版本
* claude-agent-sdk 0.2.140 或更高版本，该版本新增了本示例在失败路径上捕获的带类型的 `ResultError`
* 一个 Anthropic API 密钥（[在此获取](https://platform.claude.com/settings/keys)）

请在该笔记本自身的目录下运行它，这样它写入的文件就落在它旁边。

**成本说明：**本笔记本的实时运行完整跑一遍，API 用量通常花费约一角美元，配套脚本的运行也大致如此。每次运行都会在汇总行中打印自身成本。

## 设置

首先，安装所需依赖：

In [1]:
%%capture
%pip install -U "claude-agent-sdk>=0.2.140" python-dotenv

**说明：**请确保 `.env` 文件包含：

```bash
ANTHROPIC_API_KEY=your_key_here
```

加载你的环境变量：

In [2]:
from dotenv import load_dotenv

_ = load_dotenv()

## 创建待审阅的示例仓库

创建一个带有已知缺陷的仓库，以便核对审阅器报告了什么。下面的代码在 `demo_repo/` 中植入一个小服务，供本笔记本中的审阅使用。每个文件的源码都以字符串常量的形式出现，由代码写入磁盘。这些文件永远不会被导入，因此缺陷保持惰性；结尾的清理一节会在你完成时删除 `demo_repo/`。该服务的两个应用文件各带一个真实缺陷：

* `app/config.py` 通过复制整个环境变量来构建配置，然后打印结果。这两行合起来会把进程持有的每一个环境变量都写入日志，其中包含密钥。
* `app/math_utils.py` 通过除以 `len(values)` 来计算平均值。给它一个空列表，它就会抛出 `ZeroDivisionError`。

In [3]:
from pathlib import Path

REPO_DIR = Path("demo_repo").resolve()

README_MD = """# sample-repo

A small service kept here so the scheduled reviewer has something to read.
"""

CONFIG_PY = """import os


def load_config(path):
    config = {"source": path}
    config["environment"] = dict(os.environ)
    print("loaded config:", config)
    return config
"""

MATH_UTILS_PY = """def divide(numerator, denominator):
    return numerator / denominator


def average(values):
    return divide(sum(values), len(values))
"""

SAMPLE_REPO_FILES = {
    "README.md": README_MD,
    "app/config.py": CONFIG_PY,
    "app/math_utils.py": MATH_UTILS_PY,
}

for relative_path, body in SAMPLE_REPO_FILES.items():
    target = REPO_DIR / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(body, encoding="utf-8")

seeded = sorted(str(path.relative_to(REPO_DIR)) for path in REPO_DIR.rglob("*") if path.is_file())
print("sample repo:", seeded)

sample repo: ['README.md', 'app/config.py', 'app/math_utils.py']


## 定义审阅的返回内容

示例仓库就位后，下一块内容是审阅的答案约定。定时审阅的每个消费者都是程序，因此审阅器以一个按字段读取的 JSON 对象作答。`output_format` 接受一个 JSON 模式，响应到达 `ResultMessage.structured_output` 时已符合该模式。每次审阅都带一个审阅 id、一个来自封闭集合的结论，以及各自带 id 的发现项。

发现项 id 支撑起连续性证明。后续审阅会点名上一轮的 id，表明审阅器把这些发现项延续了下来。

In [4]:
from enum import StrEnum


class Verdict(StrEnum):
    """The two verdicts a scheduled review may report."""

    OK = "ok"
    CONCERNS = "concerns"


VERDICT_VALUES = [verdict.value for verdict in Verdict]

FINDING_SCHEMA = {
    "type": "object",
    "properties": {
        "id": {"type": "string"},
        "file": {"type": "string"},
        "summary": {"type": "string"},
    },
    "required": ["id", "file", "summary"],
}

FIRST_REVIEW_SCHEMA = {
    "type": "object",
    "properties": {
        "review_id": {"type": "string"},
        "verdict": {"type": "string", "enum": VERDICT_VALUES},
        "findings": {"type": "array", "items": FINDING_SCHEMA},
    },
    "required": ["review_id", "verdict", "findings"],
}

## 为后续审阅扩展模式

本示例会运行两次审阅，第二次需要一种回指第一次的方式。`previous_review_id` 和 `previous_finding_ids` 回指第一次审阅，`resolved` 列出不再适用的上一轮发现项 id。合并第一个模式可以把共用的部分放在一处，避免两个模式彼此偏离。

`previous_review_id` 和 `previous_finding_ids` 承载连续性断言。当审阅器把上一次审阅延续下来时，这些字段会带着第一次运行的审阅 id 和发现项 id 返回。当审阅器重新开始时，这些字段会说明这一点，你可以从该次运行的日志行得知。

In [5]:
CONTINUITY_PROPERTIES = {
    "previous_review_id": {"type": "string"},
    "previous_finding_ids": {"type": "array", "items": {"type": "string"}},
    "resolved": {"type": "array", "items": {"type": "string"}},
}

FOLLOW_UP_REVIEW_SCHEMA = FIRST_REVIEW_SCHEMA | {
    "properties": FIRST_REVIEW_SCHEMA["properties"] | CONTINUITY_PROPERTIES,
    "required": [*FIRST_REVIEW_SCHEMA["required"], *CONTINUITY_PROPERTIES.keys()],
}

print("follow-up required fields:", FOLLOW_UP_REVIEW_SCHEMA["required"])

follow-up required fields: ['review_id', 'verdict', 'findings', 'previous_review_id', 'previous_finding_ids', 'resolved']


## 配置有边界、只读的审阅器

两个模式都定义好之后，接下来是智能体配置。以下选项适合无人值守运行的智能体：

* **`tools`**：决定智能体拥有哪些内置工具。把列表限制为 `Read`、`Glob` 和 `Grep`，审阅器就没有任何写入工具。
* **`allowed_tools`**：决定哪些工具使用无需权限提示即可运行。`Read` 授权被限定在仓库路径内，在 `dontAsk` 之下，该路径之外的读取会作为工具错误被拒绝。`Glob` 和 `Grep` 以裸授权形式进入，即整个工具获批，真正约束它们的是 `deny_reads_outside_repo` 钩子。运行汇总中出现的 `StructuredOutput` 条目来自 `output_format`，而非这个列表，它只承载响应内容。
* **`permission_mode="dontAsk"`**：从不提示，并拒绝一切未经预先批准的操作。`auto` 权限模式同样无需常规提示即可运行，它在运行时由模型分类器对每次调用批准或拒绝。本审阅器使用 `dontAsk`，以使其工具面固定为允许规则。
* **`model`**：把审阅锁定到一个具名模型。不加锁定时，SDK 使用环境的默认模型，而默认模型会随时间变化。锁定让定时任务的成本和审阅行为保持在你设定的水平，直到你主动更改。
* **`strict_mcp_config=True`**：把会话限制在本配置声明的 MCP 服务器之内，也就是没有任何 MCP 服务器。若不设置，机器或账户上配置的 MCP 服务器会附加到会话中，其工具定义会进入每一次请求。
* **`max_turns` 和 `max_budget_usd`**：为智能体循环和花费设置上限。超过任一上限都会以终止性错误结果结束运行，即 `error_max_turns` 或 `error_max_budget_usd`，SDK 会以 `ResultError` 的形式把它抛给你的代码。运行是在花费已经超出预算之后才停止的，因此某个周期可能超出上限结束。
* **`hooks`**：把 `deny_reads_outside_repo` 注册为 [﷐2﷑](https://code.claude.com/docs/en/agent-sdk/hooks) 回调。该回调能看到每一次 `Read`、`Grep` 和 `Glob` 调用，并拒绝任何路径解析到仓库之外的调用，以及任何以 `/` 或 `~` 开头、带有 `..` 片段、或包含 `{` 花括号的 Glob 模式。
* **`setting_sources=[]`**：让运行不受机器上文件系统设置的影响。这样，定时审阅在你的笔记本电脑上，与在运行 cron 任务的机器上表现一致。空列表也把被审阅仓库自身的 `.claude/settings.json` 和 `CLAUDE.md` 排除在会话之外。审阅器的 `cwd` 就是那个仓库，而你不掌控的仓库不应自行配置自己的审阅器。
* **`output_format` 与 `resume`**：`output_format` 要求一个符合你传入模式的回复，`resume` 则承载跨次运行的连续性。冷启动运行时传 `None`，定时后续运行时传上一次运行的会话 id。

### 这些选项实现的护栏

上面的选项组合为实现无人值守运行智能体设置了若干护栏。只读工具授权、MCP 与设置隔离，以及回合和预算上限，限定了单次运行能做什么。定时会话重置限制了会话上下文能保留多久。

读取范围受到两次限定。在笔记本和脚本中，`allowed_tools` 授权都把 `Read` 限定在仓库路径内，并且读取仍然遵循任务用户自身的文件访问权限。请在访问权限仅限于审阅所需内容的用户下执行部署步骤。

Claude Code 尽最大努力把 `Read` 规则应用于 `Grep` 和 `Glob`。`deny_reads_outside_repo` 使读取范围限制无需依赖这种覆盖也能成立。

配套脚本安装同一个钩子，而该钩子已经拒绝对该服务目录的读取。脚本还加入一条 `disallowed_tools` 规则，作为独立的第二层，在该目录上屏蔽 `Read` 工具——环境变量文件和会话文件就存放在那里。该规则只在传入 `service_dir` 参数时才设置，而笔记本中的调用不设置它。

同样的护栏用于防范提示词注入，相关内容见[防范提示词注入](https://code.claude.com/docs/en/security#protect-against-prompt-injection)。当 `tools` 被限定为 `Read`、`Glob` 和 `Grep` 时，会话没有 shell 也没有网络工具，审阅器读到的内容除了进入回复本身之外无处可去。这就是本笔记本中答案键要检查回复的原因，也是 `review.log` 得到与被审阅仓库相同处理的原因。

In [6]:
from typing import Any

from claude_agent_sdk import (
    AssistantMessage,
    ClaudeAgentOptions,
    HookContext,
    HookMatcher,
    ResultError,
    ResultMessage,
    TextBlock,
    ToolUseBlock,
    query,
)

READ_ONLY_TOOLS = ["Read", "Glob", "Grep"]
MODEL = "claude-sonnet-5"
FIRST_RUN_BUDGET_USD = 0.25
FOLLOW_UP_BUDGET_USD = 0.20
FIRST_RUN_MAX_TURNS = 20
FOLLOW_UP_MAX_TURNS = 14
TEXT_LOG_CHARS = 400


def clip(text: Any) -> str:
    """Flatten text onto one bounded log line, marking the cut when one happens.

    Findings arrive as free text and land in a log other tools grep. Dropping
    unprintable characters and collapsing whitespace keeps each field on one
    log line of its own.
    """
    printable = "".join(char if char.isprintable() else " " for char in str(text))
    flat = " ".join(printable.split())
    return flat if len(flat) <= TEXT_LOG_CHARS else flat[:TEXT_LOG_CHARS] + "..."


REVIEWER_SYSTEM_PROMPT = (
    "You are a repository review agent running on a schedule with nobody "
    "watching. Inspect the files with your read-only tools, keep the review "
    "short, and answer with the JSON object the caller's schema describes."
)


def deny_reads_outside_repo(repo: Path):
    """Deny any Read, Grep, or Glob call whose path resolves outside repo.

    The hook confines Grep and Glob deterministically instead of relying
    on Read-rule coverage of those tools. Candidate paths resolve
    symlinks, so a Read through an in-repository link is denied when it
    lands outside. Absolute Glob patterns, patterns carrying a `..` segment, and
    patterns carrying brace syntax the segment check can't reason about are
    denied rather than trusted to stay inside their search root.
    """

    def denial(reason: str) -> dict[str, Any]:
        return {
            "hookSpecificOutput": {
                "hookEventName": "PreToolUse",
                "permissionDecision": "deny",
                "permissionDecisionReason": reason,
            }
        }

    async def deny(
        input_data: dict[str, Any], tool_use_id: str | None, context: HookContext
    ) -> dict[str, Any]:
        tool_input = input_data.get("tool_input") or {}
        raw = tool_input.get("file_path") or tool_input.get("path")
        if raw:
            candidate = Path(raw).expanduser()
            if not candidate.is_absolute():
                candidate = repo / candidate
            candidate = candidate.resolve()
            if not candidate.is_relative_to(repo):
                return denial(f"path outside the repository under review: {candidate}")
        if input_data.get("tool_name") == "Glob":
            pattern = tool_input.get("pattern")
            if isinstance(pattern, str) and (
                pattern.startswith(("/", "~")) or "{" in pattern or ".." in pattern.split("/")
            ):
                return denial(f"glob pattern may leave the repository under review: {pattern}")
        return {}

    return deny


def review_options(
    repo: Path,
    schema: dict[str, Any],
    max_turns: int,
    resume_session_id: str | None = None,
    service_dir: Path | None = None,
) -> ClaudeAgentOptions:
    """Build the options for one scheduled review pass."""
    options = ClaudeAgentOptions(
        # The agent works inside the reviewed repository.
        cwd=str(repo),
        system_prompt=REVIEWER_SYSTEM_PROMPT,
        tools=READ_ONLY_TOOLS,
        # The //-anchored form is deliberate: with a single leading slash
        # the rule anchors at the settings source, not the filesystem
        # root, matches nothing, and every read is denied under dontAsk.
        # repo is absolute, so the rendered rule keeps the double slash.
        allowed_tools=[f"Read(/{repo}/**)", "Glob", "Grep"],
        permission_mode="dontAsk",
        model=MODEL,
        # Keep machine- and account-configured MCP servers out of the
        # session.
        strict_mcp_config=True,
        max_turns=max_turns,
        max_budget_usd=FOLLOW_UP_BUDGET_USD if resume_session_id else FIRST_RUN_BUDGET_USD,
        hooks={
            "PreToolUse": [
                HookMatcher(matcher="Read|Grep|Glob", hooks=[deny_reads_outside_repo(repo)])
            ]
        },
        # Same behavior on a laptop and on the box that runs the cron job.
        setting_sources=[],
        output_format={"type": "json_schema", "schema": schema},
        # None on a cold run, and the persisted session id on a follow-up.
        resume=resume_session_id,
    )
    if service_dir is not None:
        # A second, independent layer beside the hook, which already
        # denies these reads: block the Read tool on the service
        # directory, where the env and session files live.
        options.disallowed_tools = [f"Read(/{service_dir}/**)"]
    return options

## 用回退方式读取回复

回复到达时已经按审阅模式校验过。下面的两个读取函数 `verdict_of` 和 `finding_ids`，在使用点上针对 `verdict` 和 `findings` 这两个代码据以行动的字段重复这一检查。每个读取函数匹配其字段所需的形式，否则回退到一个固定默认值。该单元格还定义了 `RunOutcome`，也就是运行器填充的容器，以及 `string_items`，即连续性检查使用的列表读取函数。

缺失或无法识别的结论按 `concerns` 读取，也就是引起关注的状态。以其他任何形式到达的发现项列表不会产出任何 id，而不是给出部分猜测。代码结尾在畸形输入上验证这两个默认值，因此下面的输出展示的是回退结果。

In [7]:
from dataclasses import dataclass, field


@dataclass
class RunOutcome:
    """What one scheduled run reported back."""

    session_id: str | None = None
    subtype: str = "no_result"
    num_turns: int = 0
    total_cost_usd: float | None = None
    denials: int = 0
    payload: dict[str, Any] = field(default_factory=dict)
    tools_attempted: list[str] = field(default_factory=list)


def sequence_of(value: Any) -> list[Any]:
    """Return a list-shaped field's items, or nothing when it isn't one."""
    match value:
        case [*items]:
            return items
        case _:
            return []


def verdict_of(payload: dict[str, Any]) -> Verdict:
    """Read the verdict from the structured reply, falling back to concerns
    when the field is missing or unrecognized."""
    match payload:
        case {"verdict": str(value)} if value in VERDICT_VALUES:
            return Verdict(value)
        case _:
            return Verdict.CONCERNS


def finding_ids(payload: dict[str, Any]) -> list[str]:
    """Collect finding ids from the structured reply."""
    ids: list[str] = []
    for finding in sequence_of(payload.get("findings")):
        match finding:
            case {"id": str(finding_id)}:
                ids.append(finding_id)
    return ids


def string_items(value: Any) -> list[str]:
    """Collect the plain strings from a list-shaped field."""
    return [item for item in sequence_of(value) if isinstance(item, str)]


print("fallbacks:", verdict_of({}), finding_ids({"findings": "not-a-list"}))

fallbacks: concerns []


## 构建审阅运行器

模式、选项与读取函数在运行器中汇合。运行器流式输出消息，打印审阅者所说的内容，列出审阅者使用的工具，并保留最后一条 `ResultMessage` 的 `session_id` 和 `structured_output`。

每次运行末尾的报告先打印结论，然后逐行打印每个发现项，都通过回复读取函数读取。每个块的打印有长度上限，被截断的行会带一个可见标志，这个上限防止一条长回复淹没定时日志。结构化输出的运行通常很少输出叙述性文字，而运行可能在工具使用之前叙述一两句。在记录下来的运行中，两个周期都只流式输出工具使用。无论哪种情况，日志中的发现项都来自回复的载荷，而非流式文本。

同一个运行器作为 [﷐0﷑](scheduled_review.py) 随笔记本一起提供，当你把审阅器排上时间计划时，部署的就是这个配套脚本。

失败的运行以非零状态退出，并且从不打印其完成行。两类失败都以这种方式结束：

* **超出上限**：`max_turns` 或 `max_budget_usd` 会以终止性错误结果结束运行，并以非零 CLI 状态退出。SDK 会把该结果以抛出的 `ResultError` 形式呈现给你的代码，其中带有该结果的 subtype。
* **没有终止性结果的失败**：运行中途被终止的 CLI 进程，或始终未能建立的连接，会抛出其自身的异常类型，这类异常逃出了类型化的捕获，因此不会打印失败行。

每次审阅运行都在其 `run_review` 调用周围捕获 `ResultError`，打印一行 `REVIEW-CYCLE-INCOMPLETE`，然后重新抛出。笔记本在失败处停止，而不是继续走到成功行。在“证明失败路径”中人工触发的运行会捕获 `ResultError` 而不重新抛出。触发失败正是那个单元格的目的。

In [8]:
async def run_review(
    *,
    repo: Path,
    label: str,
    prompt: str,
    schema: dict[str, Any],
    max_turns: int,
    resume_session_id: str | None = None,
    service_dir: Path | None = None,
) -> RunOutcome:
    """Run one review pass and return its structured result."""
    options = review_options(repo, schema, max_turns, resume_session_id, service_dir)
    outcome = RunOutcome()
    async for message in query(prompt=prompt, options=options):
        match message:
            case AssistantMessage(content=blocks):
                for block in blocks:
                    match block:
                        case TextBlock(text=text):
                            line = text.strip()
                            if line:
                                # Cap what a scheduled log absorbs per block.
                                print(f"[{label}] {clip(line)}")
                        case ToolUseBlock(name=tool_name):
                            outcome.tools_attempted.append(tool_name)
            case ResultMessage() as result:
                outcome.session_id = result.session_id
                outcome.subtype = result.subtype
                outcome.num_turns = result.num_turns
                outcome.total_cost_usd = result.total_cost_usd
                outcome.denials = len(result.permission_denials or [])
                if isinstance(result.structured_output, dict):
                    outcome.payload = result.structured_output
    return outcome


def report(label: str, outcome: RunOutcome) -> Verdict:
    """Print the scheduler-shaped lines for one run and return its verdict."""
    cost = f"{outcome.total_cost_usd:.4f}" if outcome.total_cost_usd is not None else "n/a"
    print(
        f"{label} session={outcome.session_id} subtype={outcome.subtype} "
        f"turns={outcome.num_turns} denials={outcome.denials} "
        f"cost_usd={cost} "
        f"tools_attempted={','.join(dict.fromkeys(outcome.tools_attempted)) or 'none'}"
    )
    verdict = verdict_of(outcome.payload)
    print(f"VERDICT: {verdict}")
    for finding in sequence_of(outcome.payload.get("findings")):
        match finding:
            case {"id": str(finding_id), "file": str(file_path), "summary": str(summary)}:
                print(f"  {clip(finding_id)} {clip(file_path)}: {clip(summary)}")
    return verdict

## 运行冷启动审阅

两次实时运行从这里开始，作为时间计划的前两个周期，其间间隔被压缩。第一次审阅是冷启动运行。在未设置 `resume` 的情况下，该运行读取整个仓库，并以首次审阅的模式作答。运行以结论、一组发现项 id，以及下一次运行用于恢复的 `session_id` 结束。

**预期输出：**一条带 `subtype=success` 的 `RUN-1` 汇总行，随后是 `VERDICT: concerns`，其下是覆盖两个植入缺陷的发现项行。一个缺陷可能拆成多个发现项，不过记录下来的运行对每个植入缺陷各打印一条。汇总行中的会话 id 就是运行 2 用于恢复的对象。

汇总行中非零的 `denials` 计数说明读取范围限制正在生效。该钩子拒绝以根目录为锚点的 `Read` 尝试，例如 `/app/config.py`，而审阅者会改用仓库内的路径重试。使用花括号展开的 `Glob` 模式，例如 `**/*.{py,md}`，也计入同一个计数。某次运行是否显示拒绝次数，取决于审阅者首先尝试哪些路径和模式。

拒绝会在回合预算之内产生重试成本。如果你的仓库的审阅依赖花括号模式，就调整钩子，先展开这些模式并对每次展开执行同样的检查，而不是直接拒绝该模式。

In [9]:
import uuid

FIRST_REVIEW_ID = f"review-{uuid.uuid4().hex[:8]}"
FIRST_PROMPT = (
    f"Scheduled review {FIRST_REVIEW_ID} of the repository in the current "
    "directory. "
    "Read the source files and report correctness or security problems you can "
    "point at a specific file and function. Give each finding a summary of "
    "one or two sentences and an id of the form "
    'F1, F2, and so on. Use the verdict "concerns" when you report at least one '
    'finding and "ok" when the repository looks clean. Echo '
    f"{FIRST_REVIEW_ID} back as review_id."
)

try:
    first = await run_review(
        repo=REPO_DIR,
        label="run-1",
        prompt=FIRST_PROMPT,
        schema=FIRST_REVIEW_SCHEMA,
        max_turns=FIRST_RUN_MAX_TURNS,
    )
except ResultError as exc:
    cost = (exc.data or {}).get("total_cost_usd")
    cost_str = f"{cost:.4f}" if isinstance(cost, (int, float)) else "n/a"
    print(
        f"REVIEW-CYCLE-INCOMPLETE stage=run-1 subtype={exc.subtype} "
        f"reason={exc.terminal_reason} cost_usd={cost_str}"
    )
    raise

if first.subtype != "success" or first.session_id is None or not first.payload:
    print(f"REVIEW-CYCLE-INCOMPLETE stage=run-1 subtype={first.subtype}")
    raise RuntimeError("run 1 did not complete; there is no session to resume")
first_verdict = report("RUN-1", first)

RUN-1 session=2d83a97b-27ff-403e-a067-2fb5b6474d9b subtype=success turns=10 denials=3 cost_usd=0.0278 tools_attempted=Glob,Read,StructuredOutput
VERDICT: concerns
  F1 app/math_utils.py: divide() does not guard against denominator being zero, and average() calls divide(sum(values), len(values)) which raises ZeroDivisionError when values is an empty list.
  F2 app/config.py: load_config() copies the entire process environment (os.environ) into the config dict and prints it via print(), which can leak secrets/credentials (API keys, tokens) into logs.


## 在周期之间改动仓库

在实际部署中，落在两个周期之间的改动，要么修复了审阅者报告过的某个发现项，要么引入了审阅者尚未见过的问题。下面的代码在后续运行之前各做一处这类改动。

代码修复了 `app/math_utils.py` 中的零值保护，也就是应当以已解决状态返回的那个发现项。它植入一个新文件 `app/retry.py`，其重试循环没有上限，这正是后续审阅应当作为新问题捕获的问题。这个新问题位于运行 1 从未读取的文件中。捕获它，就证明后续审阅查看的是仓库当前的样子，而不是依据它已经熟悉的文件作答。

In [10]:
MATH_UTILS_FIXED = """def divide(numerator, denominator):
    if denominator == 0:
        raise ValueError("denominator must be non-zero")
    return numerator / denominator


def average(values):
    if not values:
        return 0.0
    return divide(sum(values), len(values))
"""

RETRY_PY = """import time


def fetch_with_retry(fetch):
    while True:
        try:
            return fetch()
        except ConnectionError:
            time.sleep(1)
"""

(REPO_DIR / "app/math_utils.py").write_text(MATH_UTILS_FIXED, encoding="utf-8")
(REPO_DIR / "app/retry.py").write_text(RETRY_PY, encoding="utf-8")
changed = sorted(str(path.relative_to(REPO_DIR)) for path in REPO_DIR.rglob("*") if path.is_file())
print("repository now:", changed)

repository now: ['README.md', 'app/config.py', 'app/math_utils.py', 'app/retry.py']


## 运行恢复后的审阅

下一次定时运行把第一次运行的 `session_id` 作为 `resume` 传入，并要求使用后续审阅的模式。提示词的第一条指令是重新列出仓库的文件。没有这一步，被恢复的智能体可能依据它在运行 1 中读过的文件作答，而察觉不到 `app/retry.py` 是新文件。下面的代码把回复与运行 1 对照检查，并打印一条 `RESUME-LINK` 行，用三个字段承载连续性证明：

* `same_session`：两次运行共享同一个会话，以证明恢复本身生效
* `prior_review_id_echoed`：后续审阅点名了首次审阅的 id
* `recalled_findings`：后续审阅带回了第一次运行的多少个发现项 id

当三者都成立时，说明审阅者是依据它已经完成的审阅作答。当三者中有一个不成立时，代码会单独打印一行 `RESUME-LINK-BROKEN` 标志。回复还会列出 `resolved`，即不再适用的上一轮发现项 id。

**预期输出：**

* 一条 `RESUME-LINK` 行，其中 `same_session=True`、`prior_review_id_echoed=True`，并且 `recalled_findings` 计数完整
* 一份 `resolved` 列表，其中包含零值保护相关的发现项 id
* 保留配置泄露、并新增重试缺陷的发现项
* 用于结束该周期的 `REVIEW-CYCLE-COMPLETE` 行

In [11]:
SECOND_REVIEW_ID = f"review-{uuid.uuid4().hex[:8]}"
FOLLOW_UP_PROMPT = (
    f"Scheduled review {SECOND_REVIEW_ID} of the same repository, one cycle "
    "later. List the repository's files again, then read the source files as "
    "they stand now. Answer from the most recent review you did in this "
    "conversation: put its review id in previous_review_id and the ids of the "
    "findings it reported in previous_finding_ids. Report the findings that "
    "still apply, keeping the ids you gave them earlier, list in resolved the "
    "ids of previous findings that no longer apply, and report anything newly "
    "wrong as a new finding with a summary of one or two sentences. Echo "
    f"{SECOND_REVIEW_ID} back as review_id."
)

try:
    second = await run_review(
        repo=REPO_DIR,
        label="run-2",
        prompt=FOLLOW_UP_PROMPT,
        schema=FOLLOW_UP_REVIEW_SCHEMA,
        max_turns=FOLLOW_UP_MAX_TURNS,
        resume_session_id=first.session_id,
    )
except ResultError as exc:
    cost = (exc.data or {}).get("total_cost_usd")
    cost_str = f"{cost:.4f}" if isinstance(cost, (int, float)) else "n/a"
    print(
        f"REVIEW-CYCLE-INCOMPLETE stage=run-2 subtype={exc.subtype} "
        f"reason={exc.terminal_reason} cost_usd={cost_str}"
    )
    raise

if second.subtype != "success" or second.session_id is None or not second.payload:
    print(f"REVIEW-CYCLE-INCOMPLETE stage=run-2 subtype={second.subtype}")
    raise RuntimeError("run 2 did not complete")
second_verdict = report("RUN-2", second)

prior_ids = finding_ids(first.payload)
recalled_ids = string_items(second.payload.get("previous_finding_ids"))
matched = sorted(set(recalled_ids) & set(prior_ids))
echoed = second.payload.get("previous_review_id") == FIRST_REVIEW_ID
print(
    f"RESUME-LINK run1_session={first.session_id} "
    f"run2_session={second.session_id} "
    f"same_session={second.session_id == first.session_id} "
    f"prior_review_id_echoed={echoed} "
    f"recalled_findings={len(matched)}/{len(set(prior_ids))}"
)
print(f"resolved={[clip(i) for i in string_items(second.payload.get('resolved'))]}")
if second.session_id != first.session_id or not echoed or (prior_ids and not matched):
    # Prints on a completed run by design, mirroring the script's marker.
    print(
        f"RESUME-LINK-BROKEN same_session={second.session_id == first.session_id} "
        f"recalled_findings={len(matched)}/{len(set(prior_ids))}"
    )
print(f"REVIEW-CYCLE-COMPLETE runs=2 verdicts={first_verdict},{second_verdict}")

RUN-2 session=2d83a97b-27ff-403e-a067-2fb5b6474d9b subtype=success turns=7 denials=0 cost_usd=0.0339 tools_attempted=Glob,Read,StructuredOutput
VERDICT: concerns
  F2 app/config.py: load_config() still copies the entire process environment (os.environ) into the config dict and prints it via print(), which can leak secrets/credentials into logs.
  F3 app/retry.py: fetch_with_retry() retries in an unbounded infinite loop with a fixed 1-second sleep and no max attempt limit or exponential backoff, risking indefinite hangs/resource exhaustion if the underlying fetch keeps raising ConnectionError; other exceptions are also not handled.
RESUME-LINK run1_session=2d83a97b-27ff-403e-a067-2fb5b6474d9b run2_session=2d83a97b-27ff-403e-a067-2fb5b6474d9b same_session=True prior_review_id_echoed=True recalled_findings=2/2
resolved=['F1']
REVIEW-CYCLE-COMPLETE runs=2 verdicts=concerns,concerns


## 把发现项与植入的改动对照检查

仓库中的缺陷以及周期之间的改动事先已知，因此它们充当两次运行的答案键。

### 检查运行 1 的报告

运行 1 至少报告了植入的两个缺陷：

* `app/config.py`：加载器把整个环境变量复制进配置并打印出来
* `app/math_utils.py`：`divide` 没有零值保护，因此 `average([])` 会抛出 `ZeroDivisionError`

`config["environment"] = dict(os.environ)` 只是一行看似无害的代码。泄露之所以成立，是因为下一行的 `print` 把它发布出来，而环境变量正是部署存放密钥的地方。报告这一点意味着审阅者把贯穿该文件的这三个事实联系了起来。零值保护检验的只是审阅者是否仔细读取。

### 检查后续审阅的增量

后续审阅的报告把周期之间的改动分成三个部分：

* 配置泄露仍留在 `findings` 中，沿用其此前的 id
* 已修复的零值保护的 id 移入 `resolved`
* `app/retry.py` 中没有上限的重试作为新发现项出现

`prior_review_id_echoed=True` 意味着被恢复的审阅者点名了它在运行 1 中收到的 id，而 `recalled_findings` 统计运行 1 的多少个发现项 id 被带回。

### 预期运行之间的差异

模型输出在运行之间会有变化。审阅者可能：

* 用不同的措辞表述发现项
* 分配不同的 id
* 把一个植入的缺陷拆成多个发现项（零值保护可能作为一个发现项出现，也可能拆成 `divide` 和 `average` 两个独立发现项）
* 报告额外的低严重度观察

`resolved` 中的 id 跟随改动实际修复了什么，无论审阅者把它们如何拆分。无论措辞如何，正确的首次运行会报告两个植入的缺陷，正确的后续运行会报告三部分增量，而 `RESUME-LINK` 字段承载连续性证明。

在 `demo_repo/` 中改动别的内容并再次运行后续审阅，观察增量如何变化。笔记本把它的基线固定在运行 1，因此重复的后续运行会报告 `prior_review_id_echoed=False` 并打印 `RESUME-LINK-BROKEN` 标志，即使恢复确实生效。配套脚本在每次运行后保存新的基线，在那边重复的后续运行会保持其链接完整。

## 证明失败路径

运行器一节说明失败的运行是什么样子。下面的单元格刻意触发一次失败：在 `max_turns=1` 下针对同一个仓库运行一次全新的审阅，这个上限装不下一次完整审阅。运行以终止性错误结果结束，SDK 把它作为 `ResultError` 抛出，而 `except` 块打印带有该结果 `subtype` 和 `terminal_reason` 的失败行。

**预期输出：**一条 `REVIEW-CYCLE-INCOMPLETE` 行，其中 `stage=induced subtype=error_max_turns reason=max_turns` 并附上失败运行已消耗的成本，且没有完成行。

In [12]:
try:
    await run_review(
        repo=REPO_DIR,
        label="induced",
        prompt=FIRST_PROMPT,
        schema=FIRST_REVIEW_SCHEMA,
        max_turns=1,
    )
except ResultError as exc:
    cost = (exc.data or {}).get("total_cost_usd")
    cost_str = f"{cost:.4f}" if isinstance(cost, (int, float)) else "n/a"
    print(
        f"REVIEW-CYCLE-INCOMPLETE stage=induced subtype={exc.subtype} "
        f"reason={exc.terminal_reason} cost_usd={cost_str}"
    )
else:
    raise RuntimeError("the run completed under max_turns=1")

REVIEW-CYCLE-INCOMPLETE stage=induced subtype=error_max_turns reason=max_turns cost_usd=0.0032


## 把审阅器排上时间计划

上面两个周期是手动运行的。在时间计划下，每个周期是 [﷐0﷑](scheduled_review.py) 的一次调用，也就是本笔记本旁边的配套脚本。它从自己的服务目录运行，把待审阅的仓库作为参数，并沿用同样的模式和回复读取函数。

脚本的标志行写的是 `REVIEW-RUN`，而笔记本的周期写的是 `REVIEW-CYCLE`，笔记本的标志从不出现在定时日志中。当它的会话文件 `.last_review_session` 不在服务目录中时脚本冷启动审阅，当该文件存在时则恢复。它的 `read_state` 函数承载了这里由笔记本变量代替的持久化。

在排上时间计划之前先读一遍脚本。

在持久化的主机上运行审阅器。给任务一个文件访问权限仅限于审阅所需内容的用户，并以该用户执行下面的每一个步骤。恢复依赖于两个本地文件在周期之间留存：

* `.last_review_session`，脚本在自己的服务目录中随自身保存的会话文件
* SDK 写在 `~/.claude/projects/` 下的会话记录，或在设置了 `CLAUDE_CONFIG_DIR` 时写入该目录下的 `projects/` 目录

把服务目录放在被审阅的仓库之外。在这种布局下，干净的检出或分支切换无法删除会话文件，而且该文件从不出现在审阅者读取的文件之中。从本方案自身目录做一次试运行，`python scheduled_review.py demo_repo`，会在启动时被拒绝，并给出一行 `reason=repository-inside-service-directory` 用法提示词。服务目录位于仓库内部也会以同样方式被拒绝，给出 `reason=service-directory-inside-repository` 用法提示词。

### 为脚本设置时间计划

1. 创建带有自身环境的服务目录。笔记本的 `%pip` 安装只作用于笔记本自身的环境，而 cron 运行命令所指定的任何解释器。服务目录带有自己的虚拟环境，并安装了 SDK：

   ```bash
   sudo mkdir -p /srv/reviewer && sudo chown $USER /srv/reviewer && cd /srv/reviewer
   python3 -m venv .venv
   .venv/bin/pip install "claude-agent-sdk>=0.2.140"
   cp /path/to/scheduled_review.py .
   ```

   使用 3.11 或更高版本的 `python3`，也就是前置条件所设定的下限，因为更旧的解释器会在脚本的第一个 import 处失败。

2. 创建 cron 任务所加载的环境文件，位于 `/srv/reviewer/reviewer.env`。调度器必须提供环境，因为 cron 启动时几乎没有任何环境。使用 `export` 行，因为单纯的 `KEY=value` 赋值只设置一个 Python 进程永远不会继承的 shell 变量。Cron 会把 `PATH` 削减到最小。把它设置为覆盖包含任务所需二进制文件的目录：

   ```bash
   # /srv/reviewer/reviewer.env, sourced by the cron job
   export PATH=/usr/local/bin:$PATH
   export ANTHROPIC_API_KEY=your_key_here
   ```

   在放入密钥之前，先创建空文件并用 `touch /srv/reviewer/reviewer.env && chmod 600 /srv/reviewer/reviewer.env` 限制权限，然后再添加上面的各行。默认情况下，脚本还会拒绝自身对服务目录的 `Read` 访问，使密钥文件不进入任何发现项。同样的拒绝会让针对服务目录内任何仓库的审阅失去可见性。脚本直接拒绝那种布局。把密钥留在 crontab 行之外，因为任何能列出 crontab 的人都能读到它。

3. 从服务目录手动运行脚本一次，指向你希望被审阅的仓库：

   ```bash
   cd /srv/reviewer && . ./reviewer.env && .venv/bin/python scheduled_review.py /path/to/your/repo
   ```

   这次首次运行冷启动审阅，打印 `REVIEW-RUN-COMPLETE: cold`，并创建每一次定时运行都会恢复的 `.last_review_session` 文件和会话记录。如果运行以 1 退出并带 `error_max_budget_usd`，请在排上时间计划之前提高 `FIRST_RUN_BUDGET_USD`。装不下预算的冷启动运行永远无法建立基线，此后每个定时周期都会重复同一个失败。

4. 用 `crontab -e` 添加一条 crontab 条目，在 02：00 做夜间审阅。以执行过步骤 3 的同一个用户设置该计划，因为会话记录位于该用户的家目录下：

   ```bash
   0 2 * * * cd /srv/reviewer && . ./reviewer.env && flock -n -E 99 .review.lock .venv/bin/python scheduled_review.py /path/to/your/repo >> /srv/reviewer/review.log 2>&1 || echo "REVIEW-RUN-EXIT: $?" >> /srv/reviewer/review.log
   ```

   该条目末尾把失败调用退出状态转成一行日志。两处重定向都用绝对路径指明日志，标志永远不会落到相对于 cron 启动目录的文件中。

   `flock -n` 在上一次调用仍持有锁时跳过本次调用，而 `-E 99` 给这次跳过其自身的退出码。被跳过的运行记录 `REVIEW-RUN-EXIT: 99`，绝不会伪装成失败的运行。在 Linux 主机上 `flock` 来自 util-linux。macOS 默认没有 flock，Homebrew 的 [flock formula](https://formulae.brew.sh/formula/flock) 提供了它。

发现项会引用仓库内容，因此要给 `review.log` 与被审阅仓库相同处理。按你轮换其他服务日志的方式轮换该日志。下一节的 `REVIEW-RUN-EXIT` 行说明如何读取它。


### 读取输出行

你的调度器读取这些行：

| 行 | 触发时机 | 退出码 | 告警 |
|---|---|---|---|
| `VERDICT: ok` 或 `VERDICT: concerns` | 完成的审阅报告结构化回复的结论。失败的运行从不打印结论 | 0 | 在 `concerns` 时告警 |
| `RESUME-LINK ...` | 每一次恢复的运行。`same_session=True` 加上稳定的 `recalled_findings` 计数说明连续性正在生效，而 `current_findings` 统计后续审阅自身的发现项 | 0 | 在你自己的告警中对 `recalled_findings` 设置阈值。逐渐下降的计数不会打印任何标志，它是“按计划重置会话”一节所说明的重置信号 |
| `RESUME-LINK-BROKEN ...` | 链接彻底断裂：会话丢失、回指丢失，或召回降到零。标志位于行首，也就是用于 grep 的形式 | 0 | 告警。该运行已经为下一个周期保存了新基线 |
| `REVIEW-RUN-COMPLETE: cold` 或 `: resumed` | 一次审阅可验证地成功 | 0 | 无。它的存在意味着该周期已闭合 |
| `REVIEW-RUN-INCOMPLETE ...` | 运行以终止性错误结果结束，最常见的是超出某个上限。`subtype` 指明是什么结束了它；上限与模式失败会加上 `reason`。会话崩溃之后，该行可能打印 `cost_usd=0.0000`，这意味着成本未知而非为零 | 1 | 告警。上限失败会保留会话文件，其修复方式是配置变更 |
| `REVIEW-RUN-INCOMPLETE ... subtype=error_max_structured_output_retries` | 回复反复未通过模式校验，或一次[模型回退](https://code.claude.com/docs/en/model-config#automatic-model-fallback)在没有剩余重试可替代的情况下撤回了已完成的回复。两者都不是会话问题 | 1 | 告警。会话文件被保留 |
| `REVIEW-RUN-INCOMPLETE stage=usage ...` | 调用本身有误，例如缺少仓库参数 | 2 | 告警。修正 crontab 条目 |
| `SESSION-FILE-CLEARED: next run reviews cold` | 一次恢复的运行以 `subtype=error_during_execution` 失败，最常见的是已保存的会话不再能恢复；一次瞬时连接故障也会落到这里，用一次冷启动重建换取自愈 | 1 | 除失败告警外无。下一个周期会冷启动重建基线 |
| `SESSION-STATE-NOT-SAVED ...` | 审阅已完成但其状态无法写入，最常见的是磁盘已满或只读。标志上方的发现项是有效的。随后的 `SESSION-FILE-CLEARED` 意味着下一次运行冷启动重建；`SESSION-FILE-STALE` 意味着它将恢复上一次的会话 | 1 | 告警。修复磁盘；审阅本身无需重跑 |
| `REVIEW-RUN-EXIT: <code>` | crontab 条目末尾记录了一次非零退出，包括被跳过的重叠运行留下的 `99` | 不适用 | 除 99 之外的任何退出码都告警 |
| 完全没有该行 | 运行在终止性结果之前就已死亡，例如进程被终止或连接始终未能建立 | 非零 | 当缺少完成行且没有 `99` 跳过可解释时告警 |

一次人工触发的 `max_turns=1` 运行会打印以 `REVIEW-RUN-INCOMPLETE stage=cold subtype=error_max_turns reason=max_turns` 开头的行，也就是用于 grep 的前缀，该行以运行的 `cost_usd` 结尾。“证明失败路径”一节在笔记本中触发了同样的终止性结果。

成功的周期在日志中留下 `REVIEW-RUN-COMPLETE`，失败的周期留下 `REVIEW-RUN-EXIT` 标志。对除 99 之外的任何 `REVIEW-RUN-EXIT` 退出码告警，或对完成行缺失且没有 `99` 跳过可解释的周期告警。跳过属于正常的重叠，而反复跳过意味着运行时长超过了时间计划。所有标志的 grep 都要锚定在行首。发现项文本被裁进单行带缩进的行中，因此标志永远只位于自身行的开头。一次 `stage=resumed` 失败之后跟着 `SESSION-FILE-CLEARED` 行，意味着已保存的会话不再能恢复，最常见的原因是它在 `~/.claude/projects/` 下的记录已不存在。

### 定制审阅器

* **提示词**：让后续审阅的提示词指向审阅要跟踪的内容，例如新依赖、破坏性 API 变更或 TODO 债务。
* **模式**：添加你的告警所需的字段，例如每个发现项的严重度。回复会按你声明的任何模式校验后返回。脚本带有其自身的一份模式和读取函数副本。定制其一时，两者都要改。
* **上限**：`max_turns` 和 `max_budget_usd` 位于脚本顶部。本笔记本各单元格中的两个上限是按三文件示例仓库的规模设定的，在记录下来的运行自身 `cost_usd` 数字之上留有余量。脚本的默认值为真实仓库留有余量。预算是失控时的兜底，像回合上限那样拆分。基线审阅在 `FIRST_RUN_BUDGET_USD` 下运行，每一次后续审阅在 `FOLLOW_UP_BUDGET_USD` 下运行。后续审阅的上限才是被时间计划成倍放大的那个。按默认值，夜间任务的月度最坏情况约为 30 × $0.50 = $15，外加偶尔的冷启动重建，而记录下来的每次运行只花几美分。观察汇总行中的 `cost_usd` 数字几个周期，以找到合适的上限。该数字是客户端估算而非账单数据。[跟踪成本和用量](https://code.claude.com/docs/en/agent-sdk/cost-tracking)说明了二者差别。[﷐6﷑ 选项](https://code.claude.com/docs/en/agent-sdk/agent-loop#effort-level)是又一个成本杠杆。文档建议对只读取文件的智能体使用 `"low"`，而本审阅器就是这类智能体。
* **模型**：替换 `MODEL` 别名，以在成本与能力之间取舍。这一锁定让改动是有意的。

## 按计划重置会话

长期存活的会话按设计会积累上下文。这些上下文也会推高成本，并可能把审阅者锚定在旧结论上。按固定计划（每周，或每当仓库形态变化时）删除脚本旁边的会话文件，下一次运行就会冷启动审阅并重建基线。

后续审阅的 `recalled_findings` 计数是运维信号。当该计数下降时，会话已经退化，该重置了。

## 何时恢复、何时重新开始

定时审阅任务会恢复上一次运行的会话、每个周期重新开始，或者重新开始并喂入此前的发现项。下表把每种选择映射到它适合的任务：

| 选用 | 适用时机 |
|---|---|
| 每次调用一个新会话 | 每个工作单元都独立评判。为 pull request 把关的 CI 审阅器需要这一点：既没有旧观点，也没有被带过来的上下文左右当前判定。|
| 新会话并喂入此前的发现项 | 任务只需要上一次运行报告过的内容。一个报告上周哪些条目仍未关闭的 TODO 债务跟踪器符合这种形态：回复中的发现项 id 承载比较所需的一切，每个周期都对新代码独立评判，而且不会有已保存的会话在周期之间失效或丢失。|
| 恢复的会话 | 任务要对照上一次审阅看到的一切进行比较。一个只标识自上次审阅以来新增或变更包的夜间依赖审计，检查的是审阅者读过的完整清单，而那份清单的大部分从未进入回复。自行把它前传意味着序列化审阅者观察到的一切，而这正是恢复所省下的工作。|

## 清理

笔记本向 `demo_repo/` 写入了四个文件，即两周期夹具加上周期之间的改动。下面的代码先确认该目录中确实有夹具，然后删除它们。要先做答案键一节中的增量实验，请暂缓这次清理，需要恢复文件时稍后再运行示例仓库代码。

如果你还运行了配套脚本，也要从其服务目录删除 `.last_review_session` 文件。这些运行还在 `~/.claude/projects/` 下写入了会话记录。删除那里对应的项目文件夹即可移除它们。

In [13]:
import shutil

config_path = REPO_DIR / "app" / "config.py"
if config_path.exists() and config_path.read_text(encoding="utf-8") == CONFIG_PY:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    print("removed", REPO_DIR.name)
else:
    print("left", REPO_DIR.name, "in place: it does not hold the seeded fixture")

removed demo_repo


## 你学到了什么

* **为无人值守的智能体设界**：`tools` 和 `allowed_tools` 决定存在什么、什么无需提示词即可运行，而 `max_turns` 与 `max_budget_usd` 为循环和花费设置上限，因此定时运行只能在你选定的限度内出问题。
* **证明连续性**：恢复其上一次会话的智能体记得它已经发现了什么。让每次运行都通过在回复中回传上一次运行的发现项来证明这一点。这样，一旦恢复断裂，它立刻出现在日志中。
* **把审阅器排上时间计划**：无人值守运行的审阅是向一个程序报告，因此它的输出是调度器可以 grep 的退出码和固定行。对本审阅器而言，这意味着一行结论为仓库作答，一行完成行只在成功运行之后打印，以及一次标识所有失败的非零退出。

## 延伸阅读

* [从智能体获取结构化输出](https://code.claude.com/docs/en/agent-sdk/structured-outputs)：本方案回复所使用的 `output_format` 模式与 `structured_output` 字段的完整文档
* [使用会话](https://code.claude.com/docs/en/agent-sdk/sessions)：本方案所用恢复之外的会话生命周期，包括把一个会话分叉成两条路径
* [配置权限](https://code.claude.com/docs/en/agent-sdk/permissions)：本方案 `dontAsk` 背后的权限模式模型，以及为交互式使用设计的模式
* [安全](https://code.claude.com/docs/en/security)：Claude Code 的护栏，包括处理不可信内容的最佳实践
* [跟踪成本和用量](https://code.claude.com/docs/en/agent-sdk/cost-tracking)：`max_budget_usd` 背后以及汇总行中成本数字背后的成本面